In [1]:
# breaking
import numpy as np
from tqdm.auto import tqdm
from superconductivity.utilities.legacy.functions import bin_y_over_x
from superconductivity.models.mar import get_Imar_nA
from superconductivity.utilities.constants import G0_muS
from superconductivity.models.basics.noise import (
    apply_voltage_noise,
    make_bias_support_grid,
)
from superconductivity.utilities.functions.upsampling import upsample
from superconductivity.utilities.functions.binning import nanbin

data = np.load("breaking/fit_data.npz", allow_pickle=True)

V_mV = data["V_mV"]
x_arbu = data["x_arbu"]
tau = data["tau"]
GN_G0 = data["GN_G0"]
T_K = 0.0
Delta_meV = 0.1885
gamma_meV = 1e-6
sigmaV_mV = 26e-3

V_ = V_mV / Delta_meV
Isim_nA = np.full((x_arbu.shape[0], V_mV.shape[0]), np.nan)
Isim = np.full_like(Isim_nA, np.nan)
dGsim = np.full_like(Isim_nA, np.nan)
dRsim = np.full_like(Isim_nA, np.nan)

V_support_mV = make_bias_support_grid(V_mV, sigmaV_mV)

for i, xi_arbu in enumerate(tqdm(x_arbu)):
    tempI = get_Imar_nA(
        V_mV=V_support_mV,
        tau=tau[i, :],
        T_K=T_K,
        Delta_meV=Delta_meV,
        gamma_meV=gamma_meV,
    )
    broadened_support = apply_voltage_noise(
        V_support_mV,
        tempI,
        float(sigmaV_mV),
        order=32,
    )
    Isim_nA[i, :] = np.interp(
        V_mV,
        V_support_mV,
        broadened_support,
    )

    Isim[i, :] = Isim_nA[i, :] / (Delta_meV * G0_muS * GN_G0[i])
    dGsim[i, :] = np.gradient(Isim[i, :], V_, axis=-1)

    mask = np.isfinite(Isim[i, :])
    if np.sum(mask):
        Vsim = nanbin(
            upsample(V_[mask], axis=-1),
            upsample(Isim[i, mask], axis=-1),
            V_,
        )
        dRsim[i, :] = np.gradient(Vsim, V_)

np.savez(
    "breaking/sim.npz",
    Vbias_mV=V_mV,
    Vbias=V_,
    x_arbu=x_arbu,
    Isim_nA=Isim_nA,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    tau=tau,
    GN_G0=GN_G0,
    T_K=T_K,
    Delta_meV=Delta_meV,
    gamma_meV=gamma_meV,
    sigmaV_mV=sigmaV_mV,
)

  0%|          | 0/268 [00:00<?, ?it/s]

/var/folders/kc/8fnzl3f94vxgl8w4wm3wfvk80000gn/T/ipykernel_75049/555722599.py:53: RuntimeWarning: invalid value encountered in divide
  Isim[i, :] = Isim_nA[i, :] / (Delta_meV * G0_muS * GN_G0[i])


In [19]:
# 0.05G0, 15GHz stripline

import numpy as np
import superconductivity.api as sc

from superconductivity.utilities.meta.axis import axis
from superconductivity.utilities.meta.param import param
from superconductivity.utilities.functions.upsampling import upsample

datafit = np.load("0.05G0/15.0GHz_stripline/fit_pat.npz")
eta = datafit["eta"]
GN_G0 = np.nanmedian(datafit["GN_G0"])
Delta_meV = 0.191
sigmaV_mV = 0.025
nu_GHz = 15.0
T_K = 0.0
gamma_meV = 1e-9

Vbias0 = np.linspace(-6, 6, 3001)
Vbias = np.linspace(-0.5, 4.5, 501)
Ibias = np.linspace(-0.5, 4.5, 501)
Abias = np.linspace(0.0, 10.0, 101)

outdir = "0.05G0/15.0GHz_stripline/sim.npz"

Vbias0_mV = Vbias0 * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz

# actual simulation
sim = sc.sim_bcs(
    V_mV=axis("V_mV", values=Vbias0_mV),
    GN_G0=param("GN_G0", GN_G0),
    T_K=param("T_K", T_K),
    Delta_meV=param("Delta_meV", Delta_meV),
    gamma_meV=param("gamma_meV", gamma_meV),
    sigmaV_mV=param("sigmaV_mV", sigmaV_mV),
    nu_GHz=param("nu_GHz", nu_GHz),
    A_mV=axis("A_mV", values=Abias_mV),
)

Isim0_nA = sim["I_nA"]
Isim0 = Isim0_nA / (GN_G0 * sc.G0_muS * Delta_meV)

Vbias_mV = Vbias * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz
Ibias_nA = Ibias * GN_G0 * sc.G0_muS * Delta_meV

# binning
Isim = sc.bin(
    z=upsample(Isim0),
    x=upsample(Vbias0),
    xbins=Vbias,
    axis=1,
)
dGsim = np.gradient(Isim, Vbias, axis=1)
# dGsim = savgol_filter(dGsim, window_length=5, polyorder=0)

Vsim = sc.bin(
    z=upsample(Vbias0),
    x=upsample(Isim0),
    xbins=Ibias,
    axis=1,
)
dRsim = np.gradient(Vsim, Ibias, axis=1)

# saving progress
np.savez_compressed(
    outdir,
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

In [ ]:
# 1.8G0, 15.0GHz, stripline
import numpy as np

path = "1.8G0/15.0GHz_stripline"
data_mar = np.load(f"{path}/mar_data.npz")
data_eta = np.load(f"{path}/eta_data.npz")

V_mV = np.linspace(-0.5, 4.5, 501)
A_mV = data_eta["Abias_mV"]

tau = data_mar["tau"]
nu_GHz = data_eta["nu_GHz"]

Delta_meV = data_mar["Delta_meV"]
sigmaV_mV = data_mar["sigmaV_mV"]
gamma_meV = data_mar["gamma_meV"]
T_K = data_mar["T_K"]

sc.get_Imar_nA(
    V_mV=V_mV,
    tau=tau,
    T_K=T_K,
    Delta_meV=Delta_meV,
    gamma_meV=gamma_meV,
    charge_resolved=True,
    tau_resolved=True,
)